# ViViT on SSv2 — Full Finetuning
Target: reproduce SSv2 results. Full model finetuning (all layers unfrozen).

In [ ]:
# CELL 1 — Install (run once, then restart runtime)
!pip install transformers==4.35.0 einops -q

In [ ]:
# CELL 2 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# CELL 3 — Extract videos + copy to local disk
import os

os.makedirs('/content/ssv2/videos', exist_ok=True)
os.makedirs('/content/videos_local', exist_ok=True)

# Extract part 00 (gzip format)
if not os.path.exists('/content/ssv2/videos/20bn-something-something-v2'):
    print('Extracting...')
    os.system('tar -xzf /content/drive/MyDrive/20bn-something-something-v2-00 -C /content/ssv2/videos/')
    print('Extracted!')

# Copy to local SSD for faster reading
if not os.path.exists('/content/videos_local/20bn-something-something-v2'):
    print('Copying to local disk...')
    os.system('cp -r /content/ssv2/videos/20bn-something-something-v2 /content/videos_local/')
    print('Copied!')

VIDEO_DIR = '/content/videos_local/20bn-something-something-v2/'
print(f'Videos available: {len(os.listdir(VIDEO_DIR))}')

Extracting...
Extracted!
Copying to local disk...
Copied!
Videos available: 113442


In [ ]:
# CELL 4 — Imports
import os, json, random, re
import numpy as np
import torch
import torch.nn as nn
import cv2
from torch.utils.data import Dataset, DataLoader
from transformers import VivitForVideoClassification
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [ ]:
# CELL 5 — Load labels
LABELS_DIR = '/content/drive/MyDrive/labels'

with open(f'{LABELS_DIR}/labels.json') as f:
    labels_raw = json.load(f)
with open(f'{LABELS_DIR}/validation.json') as f:
    val_data = json.load(f)
with open(f'{LABELS_DIR}/train.json') as f:
    train_data = json.load(f)

# labels.json format: {"label name": "0", ...}
label2id = {k: int(v) for k, v in labels_raw.items()}
id2label = {int(v): k for k, v in labels_raw.items()}

print(f'Classes:          {len(label2id)}')
print(f'Validation clips: {len(val_data)}')
print(f'Train clips:      {len(train_data)}')

Classes:          174
Validation clips: 24777
Train clips:      168913


In [ ]:
# CELL 6 — Dataset + video loader

def template_to_label(template):
    return re.sub(r'\[.*?\]', 'something', template).strip()

def load_video_cv2(path, num_frames=32):
    try:
        cap   = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total <= 0:
            cap.release()
            return None
        indices = np.linspace(0, total - 1, num_frames, dtype=int)
        frames  = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret:
                continue  # skip bad frames
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (224, 224))
            frames.append(frame)
        cap.release()
        if len(frames) < num_frames // 2:
            return None
        while len(frames) < num_frames:  # pad if needed
            frames.append(frames[-1])
        return np.stack(frames[:num_frames])  # (T, H, W, C)
    except:
        return None

class SSv2Dataset(Dataset):
    def __init__(self, data, label2id, video_dir, num_frames=32):
        self.video_dir  = video_dir
        self.num_frames = num_frames
        self.valid      = []
        self.MEAN = np.array([0.485, 0.456, 0.406])
        self.STD  = np.array([0.229, 0.224, 0.225])

        for item in data:
            label_name = template_to_label(item['template'])
            label_id   = label2id.get(label_name, -1)
            if label_id == -1: continue
            for ext in ['.webm', '.mp4', '']:
                path = os.path.join(video_dir, str(item['id']) + ext)
                if os.path.exists(path):
                    self.valid.append((path, label_id))
                    break

        print(f'Valid videos: {len(self.valid)} / {len(data)}')

    def __len__(self):
        return len(self.valid)

    def __getitem__(self, idx):
        path, label_id = self.valid[idx]
        frames = load_video_cv2(path, self.num_frames)
        if frames is None:
            return None
        frames = frames.astype(np.float32) / 255.0
        frames = (frames - self.MEAN) / self.STD
        pixel_values = torch.from_numpy(frames).permute(0, 3, 1, 2).float()  # (T,C,H,W)
        return pixel_values, label_id

def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch: return None, None
    return torch.stack([b[0] for b in batch]), torch.tensor([b[1] for b in batch])

# Quick sanity check
VIDEO_DIR = '/content/videos_local/20bn-something-something-v2/'
test_ds = SSv2Dataset(val_data[:20], label2id, VIDEO_DIR, num_frames=32)
item = test_ds[0]
print(f'Item shape: {item[0].shape if item is not None else None}')  # should be torch.Size([32, 3, 224, 224])
print(f'Label: {item[1]} = {id2label[item[1]] if item is not None else None}')

Valid videos: 10 / 20
Item shape: torch.Size([32, 3, 224, 224])
Label: 140 = Spinning something that quickly stops spinning


In [ ]:
# CELL 7 — Load ViViT + replace head
torch.cuda.empty_cache()

print('Loading ViViT...')
vivit_model = VivitForVideoClassification.from_pretrained('google/vivit-b-16x2-kinetics400')

# Replace K400 head (400 classes) with SSv2 head (174 classes)
in_features = vivit_model.classifier.in_features
vivit_model.classifier = nn.Linear(in_features, 174)
nn.init.xavier_uniform_(vivit_model.classifier.weight)
nn.init.zeros_(vivit_model.classifier.bias)

vivit_model = vivit_model.to(device)
total_params = sum(p.numel() for p in vivit_model.parameters()) / 1e6
print(f'ViViT loaded — {total_params:.1f}M params')
print(f'Head: Linear({in_features}, 174)')

Loading ViViT...


ViViT loaded — 88.8M params
Head: Linear(768, 174)


In [ ]:
from transformers import VivitForVideoClassification
import torch.nn as nn

vivit_model = VivitForVideoClassification.from_pretrained('google/vivit-b-16x2-kinetics400')
vivit_model.classifier = nn.Linear(vivit_model.classifier.in_features, 174)
vivit_model.load_state_dict(torch.load('/content/drive/MyDrive/vivit_head_finetuned_full.pth',
                                        map_location='cpu', weights_only=False))
vivit_model = vivit_model.to(device)
vivit_model.eval()
print('Model loaded!')

Model loaded!


In [ ]:
# Reload fresh model
from transformers import VivitForVideoClassification
import torch.nn as nn

torch.cuda.empty_cache()

vivit_model = VivitForVideoClassification.from_pretrained('google/vivit-b-16x2-kinetics400')
vivit_model.classifier = nn.Linear(vivit_model.classifier.in_features, 174)
nn.init.xavier_uniform_(vivit_model.classifier.weight)
nn.init.zeros_(vivit_model.classifier.bias)
vivit_model = vivit_model.to(device)

# Freeze backbone, only train head
for p in vivit_model.parameters():
    p.requires_grad = False
for p in vivit_model.classifier.parameters():
    p.requires_grad = True

# All training data
VIDEO_DIR = '/content/videos_local/20bn-something-something-v2/'
random.shuffle(train_data)
train_dataset = SSv2Dataset(train_data[:40000], label2id, VIDEO_DIR, num_frames=32)
train_loader  = DataLoader(train_dataset, batch_size=8, shuffle=True,
                            num_workers=2, collate_fn=collate_fn)

optimizer = torch.optim.Adam(vivit_model.classifier.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# 5 epochs
for epoch in range(5):
    vivit_model.train()
    loss_sum, correct, total = 0, 0, 0
    for pixels, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/5'):
        if pixels is None: continue
        pixels, labels = pixels.to(device), labels.to(device)
        logits = vivit_model(pixel_values=pixels).logits
        loss   = criterion(logits, labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        loss_sum += loss.item()
        correct  += (logits.argmax(-1) == labels).sum().item()
        total    += labels.size(0)
    if total > 0:
        print(f'Epoch {epoch+1} | Loss: {loss_sum/len(train_loader):.4f} | Acc: {100*correct/total:.2f}%')

torch.save(vivit_model.state_dict(), '/content/drive/MyDrive/vivit_head_5epochs.pth')
print('Saved!')

Valid videos: 18613 / 40000


Epoch 1/5: 100%|██████████| 2327/2327 [1:36:35<00:00,  2.49s/it]


Epoch 1 | Loss: 4.0962 | Acc: 13.95%


Epoch 2/5: 100%|██████████| 2327/2327 [1:36:09<00:00,  2.48s/it]


Epoch 2 | Loss: 2.8498 | Acc: 31.26%


...

Epoch 3 | Loss: 2.3486 | Acc: 40.98%


Epoch 4/5: 100%|██████████| 2327/2327 [1:36:32<00:00,  2.49s/it]


Epoch 4 | Loss: 2.0455 | Acc: 47.00%


Epoch 5/5: 100%|██████████| 2327/2327 [1:36:21<00:00,  2.48s/it]


Epoch 5 | Loss: 1.8146 | Acc: 52.94%
Saved!


In [ ]:
# CELL 9 — Evaluate on validation set
VIDEO_DIR = '/content/videos_local/20bn-something-something-v2/'

val_dataset = SSv2Dataset(val_data, label2id, VIDEO_DIR, num_frames=32)
val_loader  = DataLoader(val_dataset, batch_size=8, shuffle=False,
                          num_workers=2, collate_fn=collate_fn)

vivit_model.eval()
top1_correct, top5_correct, total = 0, 0, 0
per_class_correct = {i: 0 for i in range(174)}
per_class_total   = {i: 0 for i in range(174)}

with torch.no_grad():
    for pixels, labels in tqdm(val_loader, desc='Evaluating'):
        if pixels is None: continue
        pixels, labels = pixels.to(device), labels.to(device)
        logits     = vivit_model(pixel_values=pixels).logits
        top1_preds = logits.argmax(-1)
        top1_correct += (top1_preds == labels).sum().item()
        top5_preds = logits.topk(5, dim=-1).indices
        for i, lbl in enumerate(labels):
            if lbl.item() in top5_preds[i].tolist(): top5_correct += 1
            per_class_total[lbl.item()]   += 1
            if top1_preds[i].item() == lbl.item(): per_class_correct[lbl.item()] += 1
        total += labels.size(0)

top1 = 100 * top1_correct / total
top5 = 100 * top5_correct / total
print(f'\n=== ViViT SSv2 Results ===')
print(f'Top-1: {top1:.2f}%  (paper: 65.9%)')
print(f'Top-5: {top5:.2f}%  (paper: 89.9%)')
print(f'Clips: {total}')

import json
with open('/content/drive/MyDrive/vivit_results.json', 'w') as f:
    json.dump({'top1': top1, 'top5': top5, 'total': total}, f, indent=2)
print('Results saved to Drive')

Valid videos: 11313 / 24777


Evaluating: 100%|██████████| 1415/1415 [59:36<00:00,  2.53s/it]


=== ViViT SSv2 Results ===
Top-1: 14.97%  (paper: 65.9%)
Top-5: 37.48%  (paper: 89.9%)
Clips: 11285
Results saved to Drive


In [ ]:
# CELL 10 — Per-class analysis
per_class_acc = {i: per_class_correct[i]/per_class_total[i]
                 for i in range(174) if per_class_total[i] > 0}
sorted_acc = sorted(per_class_acc.items(), key=lambda x: x[1])

print('=== 10 Worst Classes ===')
for cid, acc in sorted_acc[:10]:
    print(f'  {id2label[cid]:<55} {acc*100:.1f}%')

print('\n=== 10 Best Classes ===')
for cid, acc in sorted_acc[-10:]:
    print(f'  {id2label[cid]:<55} {acc*100:.1f}%')

=== 10 Worst Classes ===
  Failing to put something into something because something does not fit 0.0%
  Letting something roll up a slanted surface, so it rolls back down 0.0%
  Lifting a surface with something on it but not enough for it to slide down 0.0%
  Moving something and something so they collide with each other 0.0%
  Moving something and something so they pass each other  0.0%
  Moving something away from the camera                   0.0%
  Picking something up                                    0.0%
  Poking a stack of something without the stack collapsing 0.0%
  Poking something so that it spins around                0.0%
  Pretending or failing to wipe something off of something 0.0%

=== 10 Best Classes ===
  Pouring something into something until it overflows     31.0%
  Bending something until it breaks                       31.8%
  Plugging something into something but pulling it right out as you remove your hand 34.1%
  Throwing something in the air and catching it